# Chargement du dataset

In [33]:
import numpy as np # linear algebra
import pandas as pd

In [34]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("chandramoulinaidu/spam-classification-for-basic-nlp")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'spam-classification-for-basic-nlp' dataset.
Path to dataset files: /kaggle/input/spam-classification-for-basic-nlp


In [35]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spam-classification-for-basic-nlp/Spam Email raw text for NLP.csv


In [36]:
data = pd.read_csv('/kaggle/input/spam-classification-for-basic-nlp/Spam Email raw text for NLP.csv')
data

,CATEGORY,MESSAGE,FILE_NAME
0,1,"Dear Homeowner,\n\n \n\nInterest Rates are at ...",00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,ATTENTION: This is a MUST for ALL Computer Use...,00373.ebe8670ac56b04125c25100a36ab0510
2,1,This is a multi-part message in MIME format.\n...,00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,IMPORTANT INFORMATION:\n\n\n\nThe new domain n...,00210.050ffd105bd4e006771ee63cabc59978
4,1,This is the bottom line. If you can GIVE AWAY...,00033.9babb58d9298daa2963d4f514193d7d6
...,...,...,...
5791,0,"I'm one of the 30,000 but it's not working ver...",00609.dd49926ce94a1ea328cce9b62825bc97
5792,0,Damien Morton quoted:\n\n>W3C approves HTML 4 ...,00957.e0b56b117f3ec5f85e432a9d2a47801f
5793,0,"On Mon, 2002-07-22 at 06:50, che wrote:\n\n\n\...",01127.841233b48eceb74a825417d8d918abf8
5794,0,"Once upon a time, Manfred wrote :\n\n\n\n> I w...",01178.5c977dff972cd6eef64d4173b90307f0


# Prétraitement des données

- la Suppression des balises HTML

In [37]:
from bs4 import BeautifulSoup
def remove_html_tags_bs4(text):
    return BeautifulSoup(text, "html.parser").get_text()
data['MESSAGE'] = data['MESSAGE'].apply(remove_html_tags_bs4)
display(data.head())

,CATEGORY,MESSAGE,FILE_NAME
0,1,"Dear Homeowner,\n\n \n\nInterest Rates are at ...",00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,ATTENTION: This is a MUST for ALL Computer Use...,00373.ebe8670ac56b04125c25100a36ab0510
2,1,This is a multi-part message in MIME format.\n...,00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,IMPORTANT INFORMATION:\n\n\n\nThe new domain n...,00210.050ffd105bd4e006771ee63cabc59978
4,1,This is the bottom line. If you can GIVE AWAY...,00033.9babb58d9298daa2963d4f514193d7d6


- La Suppression des caractères spéciaux

In [38]:
import re

def remove_special_characters(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['MESSAGE'] = data['MESSAGE'].apply(remove_special_characters)
display(data.head())

,CATEGORY,MESSAGE,FILE_NAME
0,1,Dear Homeowner Interest Rates are at their low...,00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,ATTENTION This is a MUST for ALL Computer User...,00373.ebe8670ac56b04125c25100a36ab0510
2,1,This is a multipart message in MIME format Nex...,00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,IMPORTANT INFORMATION The new domain names are...,00210.050ffd105bd4e006771ee63cabc59978
4,1,This is the bottom line If you can GIVE AWAY C...,00033.9babb58d9298daa2963d4f514193d7d6


- La tokenisation du texte

In [39]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [40]:
import nltk
from nltk.tokenize import word_tokenize

try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    nltk.download('punkt')

data['MESSAGE'] = data['MESSAGE'].apply(word_tokenize)
display(data.head())

,CATEGORY,MESSAGE,FILE_NAME
0,1,"[Dear, Homeowner, Interest, Rates, are, at, th...",00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,"[ATTENTION, This, is, a, MUST, for, ALL, Compu...",00373.ebe8670ac56b04125c25100a36ab0510
2,1,"[This, is, a, multipart, message, in, MIME, fo...",00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,"[IMPORTANT, INFORMATION, The, new, domain, nam...",00210.050ffd105bd4e006771ee63cabc59978
4,1,"[This, is, the, bottom, line, If, you, can, GI...",00033.9babb58d9298daa2963d4f514193d7d6


- La conversion du texte en minuscules

In [41]:
data['MESSAGE'] = data['MESSAGE'].apply(lambda x: [word.lower() for word in x])
display(data.head())

,CATEGORY,MESSAGE,FILE_NAME
0,1,"[dear, homeowner, interest, rates, are, at, th...",00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,"[attention, this, is, a, must, for, all, compu...",00373.ebe8670ac56b04125c25100a36ab0510
2,1,"[this, is, a, multipart, message, in, mime, fo...",00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,"[important, information, the, new, domain, nam...",00210.050ffd105bd4e006771ee63cabc59978
4,1,"[this, is, the, bottom, line, if, you, can, gi...",00033.9babb58d9298daa2963d4f514193d7d6


- La suppression de la ponctuation

In [42]:
import string

def remove_punctuation(tokens):
    return [word for word in tokens if word not in string.punctuation]

data['MESSAGE'] = data['MESSAGE'].apply(remove_punctuation)
display(data.head())

,CATEGORY,MESSAGE,FILE_NAME
0,1,"[dear, homeowner, interest, rates, are, at, th...",00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,"[attention, this, is, a, must, for, all, compu...",00373.ebe8670ac56b04125c25100a36ab0510
2,1,"[this, is, a, multipart, message, in, mime, fo...",00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,"[important, information, the, new, domain, nam...",00210.050ffd105bd4e006771ee63cabc59978
4,1,"[this, is, the, bottom, line, if, you, can, gi...",00033.9babb58d9298daa2963d4f514193d7d6


- La suppression des mots vides (stopwords)

In [43]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
data['MESSAGE'] = data['MESSAGE'].apply(lambda x: [word for word in x if word not in stop_words])
display(data.head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,CATEGORY,MESSAGE,FILE_NAME
0,1,"[dear, homeowner, interest, rates, lowest, poi...",00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,"[attention, must, computer, users, newspecial,...",00373.ebe8670ac56b04125c25100a36ab0510
2,1,"[multipart, message, mime, format, nextpart000...",00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,"[important, information, new, domain, names, f...",00210.050ffd105bd4e006771ee63cabc59978
4,1,"[bottom, line, give, away, cds, free, people, ...",00033.9babb58d9298daa2963d4f514193d7d6


- Lemmatisation des mots

In [44]:
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

data['MESSAGE'] = data['MESSAGE'].apply(lemmatize_words)
display(data.head())

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,CATEGORY,MESSAGE,FILE_NAME
0,1,"[dear, homeowner, interest, rate, lowest, poin...",00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,"[attention, must, computer, user, newspecial, ...",00373.ebe8670ac56b04125c25100a36ab0510
2,1,"[multipart, message, mime, format, nextpart000...",00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,"[important, information, new, domain, name, fi...",00210.050ffd105bd4e006771ee63cabc59978
4,1,"[bottom, line, give, away, cd, free, people, l...",00033.9babb58d9298daa2963d4f514193d7d6


# Word embeddings

- Bag of Words